In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re

# ------------------ SETTINGS ------------------
MIXER = {
    "SIMALIGN_THRESHOLD": 0.12,
    "RESCUE_THRESHOLD": 0.12,
    "MANY_TO_MANY_RATIO": 0.85, # Allow extra links if they are within this ratio of best link
}

print("Loading High-Performance Models...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LaBSE for semantic word similarity
sim_model = SentenceTransformer('sentence-transformers/LaBSE').to(device)

# IndicBERT v2 for structural context alignment
awesome_id = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
awesome_tokenizer = AutoTokenizer.from_pretrained(awesome_id)
awesome_model = AutoModel.from_pretrained(awesome_id, output_hidden_states=True).to(device)
awesome_model.eval()
print(f"Models Ready (using {device}).")

Loading High-Performance Models...
Models Ready (using cpu).


In [ ]:
def calculate_aer(gold_str, pred_links, src_len, tgt_len):
    """
    Calculates Precision, Recall, and Alignment Error Rate (AER).
    Gold format: '0-0 1-1 1-2' (space separated srcIdx-tgtIdx)
    """
    if not gold_str or pd.isna(gold_str):
        return None, None, None

    try:
        sure_links = set()
        for pair in str(gold_str).strip().split():
            if '-' in pair:
                s, t = map(int, pair.split('-'))
                sure_links.add((s, t))

        unique_preds = set((i, j) for i, j, _ in pred_links)

        TP = len(sure_links.intersection(unique_preds))

        precision = TP / len(unique_preds) if len(unique_preds) > 0 else 0.0
        recall = TP / len(sure_links) if len(sure_links) > 0 else 0.0

        aer = 1 - (2 * TP) / (len(unique_preds) + len(sure_links)) if (len(unique_preds) + len(sure_links)) > 0 else 0.0

        return precision, recall, aer
    except:
        return None, None, None

In [ ]:
def run_pipeline(eng, tgt):
    # 0. Pre-cleaning
    def clean(t): return re.sub(r'[।\?\.\!\,]', '', t).strip()
    e_words = clean(eng).split()
    t_words = clean(tgt).split()

    if not e_words or not t_words: return e_words, t_words, []

    # 1. Semantic Similarity Matrix (LaBSE)
    e_emb = sim_model.encode(e_words, convert_to_tensor=True).cpu().numpy()
    t_emb = sim_model.encode(t_words, convert_to_tensor=True).cpu().numpy()
    sim_mat = cosine_similarity(e_emb, t_emb)

    # 2. Structural Alignment (IndicBERT v2 Layer 8)
    inp = awesome_tokenizer([eng, tgt], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = awesome_model(**inp)

    layer_8 = out.hidden_states[8]
    src_tokens = awesome_tokenizer.tokenize(eng)
    tgt_tokens = awesome_tokenizer.tokenize(tgt)

    def get_word_map(tokens):
        ids, curr = [], -1
        for tok in tokens:
            if tok.startswith(" "): curr += 1
            elif curr == -1: curr = 0
            ids.append(curr)
        return ids

    src_map, tgt_map = get_word_map(src_tokens), get_word_map(tgt_tokens)

    s_emb = layer_8[0, 1:len(src_tokens)+1]
    t_emb_feat = layer_8[1, 1:len(tgt_tokens)+1]
    dot = torch.matmul(s_emb, t_emb_feat.t())
    a_probs = ((torch.nn.functional.softmax(dot, dim=-1) + torch.nn.functional.softmax(dot, dim=-2)) / 2).cpu().numpy()

    # 3. Consensus Logic (Many-to-Many)
    final_links = []

    # Part A: Semantic Matches
    for i in range(len(e_words)):
        max_val = np.max(sim_mat[i])
        if max_val > MIXER['SIMALIGN_THRESHOLD']:
            for j in range(len(t_words)):
                if sim_mat[i, j] >= (max_val * MIXER['MANY_TO_MANY_RATIO']) and sim_mat[i, j] > MIXER['SIMALIGN_THRESHOLD']:
                    final_links.append((i, j, sim_mat[i, j]))

    # Part B: Structural Matches
    for i in range(min(len(src_tokens), a_probs.shape[0])):
        for j in range(min(len(tgt_tokens), a_probs.shape[1])):
            if a_probs[i, j] > 0.35:
                wi, wj = src_map[i], tgt_map[j]
                if wi < len(e_words) and wj < len(t_words):
                    final_links.append((wi, wj, a_probs[i, j]))

    return e_words, t_words, final_links

In [ ]:
# --- EXCEL CONFIGURATION ---
file_path = '/content/Eco-1 corrections.xlsx'
english_col = 'English transcription'
telugu_col = 'Correction'
gold_col = 'Gold Alignment'
MAX_TOKENS = 510

try:
    df = pd.read_excel(file_path)
    df = df.dropna(subset=[english_col, telugu_col])

    total_stats = []
    skipped_rows = []

    print(f"Processing {len(df)} rows...")

    for idx, row in df.iterrows():
        try:
            e_txt, t_txt = str(row[english_col]), str(row[telugu_col])

            # Check token length beforehand
            e_len = len(awesome_tokenizer.encode(e_txt, add_special_tokens=False))
            t_len = len(awesome_tokenizer.encode(t_txt, add_special_tokens=False))

            if e_len > MAX_TOKENS or t_len > MAX_TOKENS:
                print(f"Row {idx+1} skipped: Sentence exceeds {MAX_TOKENS} tokens.")
                skipped_rows.append(idx+1)
                continue

            e_words, t_words, raw_aligns = run_pipeline(e_txt, t_txt)

            # Deduplicate alignments
            best_aligns = {}
            for i, j, s in raw_aligns:
                if (i, j) not in best_aligns or s > best_aligns[(i, j)]:
                    best_aligns[(i, j)] = s

            sorted_aligns = sorted(best_aligns.items(), key=lambda x: x[0][0])
            aligned_e = set(idx[0] for idx in best_aligns.keys())
            aligned_t = set(idx[1] for idx in best_aligns.keys())

            null_e = [e_words[i] for i in range(len(e_words)) if i not in aligned_e]
            null_t = [t_words[j] for j in range(len(t_words)) if j not in aligned_t]

            p, r, aer = None, None, None
            if gold_col in df.columns:
                metrics_aligns = [(i, j, s) for (i, j), s in best_aligns.items()]
                p, r, aer = calculate_aer(row[gold_col], metrics_aligns, len(e_words), len(t_words))

            avg_conf = np.mean(list(best_aligns.values())) if best_aligns else 0.0

            print(f"--- Row {idx+1} ---")
            print(f"EN: {e_txt[:60]}...")
            print(f"TE: {t_txt[:60]}...")

            mapping_strs = [f"{e_words[i]} ↔ {t_words[j]}" for (i, j), _ in sorted_aligns]
            print(f"MAPPINGS: {', '.join(mapping_strs)}")

            if null_e: print(f"NULL (EN): {', '.join(null_e)}")
            if null_t: print(f"NULL (TE): {', '.join(null_t)}")

            if aer is not None:
                print(f"Metrics: Precision={p:.2f}, Recall={r:.2f}, AER={aer:.2f}")
                total_stats.append({'aer': aer, 'p': p, 'r': r})
            else:
                print(f"Quality (Avg Conf): {avg_conf:.2f}")
                total_stats.append({'conf': avg_conf})

        except Exception as row_error:
            print(f"Row {idx+1} skipped due to error: {row_error}")
            skipped_rows.append(idx+1)
            continue

    print("\n" + "="*40)
    print("            FINAL EVALUATION")
    print("="*40)
    if total_stats and 'aer' in total_stats[0]:
        valid_aer = [s['aer'] for s in total_stats if s['aer'] is not None]
        if valid_aer:
            avg_aer = np.mean(valid_aer)
            print(f"Overall AER (Accuracy Error): {avg_aer:.4f}")
            print(f"Overall Accuracy (1-AER):     {(1-avg_aer):.4f}")
    else:
        avg_score = np.mean([s['conf'] for s in total_stats]) if total_stats else 0.0
        print(f"Overall Mean Model Confidence: {avg_score:.4f}")

    if skipped_rows:
        print(f"Total rows skipped: {len(skipped_rows)} (Rows: {skipped_rows})")

except Exception as e:
    print(f"System Error: {e}")

Processing 10 rows...
--- Row 1 ---
EN: so we just took a look at a few numbers here  I thought we'l...
TE: ఇప్పుడు మనం కొన్ని నెంబర్లు చూసాం కదా… ఇప్పుడు వాటితో పాటు ప...
MAPPINGS: so ↔ అలా, so ↔ కాబట్టి, so ↔ ఇప్పుడు, we ↔ మనం, we ↔ మనకి, we ↔ మనం, we ↔ మనం, we ↔ మనం, we ↔ మనం, we ↔ మనం, we ↔ మనం, just ↔ ఇప్పుడు, just ↔ కొన్ని, just ↔ ఇప్పుడు, just ↔ పాటు, just ↔ కూడా, just ↔ అలా, just ↔ బాగా, just ↔ ఒక, just ↔ నాటికి, just ↔ అని, just ↔ ఒక, just ↔ ఎంత, just ↔ అని, just ↔ గా, just ↔ ఎంత, just ↔ ఉందో, just ↔ ఎంత, just ↔ ఉందో, just ↔ కానీ, just ↔ అని, just ↔ అది, just ↔ కాబట్టి, just ↔ ఇప్పుడు, just ↔ చాలా, just ↔ అయినదే, just ↔ కొంచెం, just ↔ కూడా, just ↔ ఎందుకంటే, just ↔ ఒక, just ↔ ఒక, just ↔ దాని, just ↔ కొన్ని, just ↔ ఒక, took ↔ తీసుకుని, a ↔ ఒక, a ↔ ఒక, a ↔ ఒక, a ↔ ఒక, a ↔ ఒక, look ↔ చూసాం, look ↔ చూసేద్దాం, look ↔ చూస్తే, look ↔ చూడవచ్చు, look ↔ చూసిన, at ↔ కి, at ↔ లో, a ↔ ఒక, a ↔ ఒక, a ↔ ఒక, a ↔ ఒక, a ↔ ఒక, few ↔ కొన్ని, few ↔ కొంచెం, few ↔ కొన్ని, numbers ↔ నెంబర్లు, numbers ↔